# Fine-tuning DistilBERT for Humor-Score Regression

Trains DistilBERT to predict the rJokes 0–11 humor score, on Google Colab's
free T4 GPU. Beat the local TF-IDF baseline of **Spearman ≈ 0.363**.

**Before you run:** set the runtime to GPU —
`Runtime → Change runtime type → T4 GPU`.

Fill in the two config values in the first cell (your GitHub repo URL and your
Hugging Face repo name), then `Runtime → Run all`.

In [ ]:
# ── CONFIG — edit these two lines ────────────────────────────────────
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/humor-intelligence.git"
HF_MODEL_REPO   = "YOUR_HF_USERNAME/humor-intelligence-distilbert"  # will be created

# training knobs (defaults are tuned for the free T4 + full 339k rows)
MODEL_NAME  = "distilbert-base-uncased"
MAX_LENGTH  = 128     # tokens per joke; longer = slower. 128 covers most jokes.
EPOCHS      = 2       # 2 is a good time/quality balance on the free tier
BATCH_SIZE  = 32
LR          = 2e-5
# ─────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Check we actually have a GPU (if this says CPU, fix the runtime type)
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (!)")

In [ ]:
# 2. Install pinned deps (Colab has torch already; add the HF stack)
!pip install -q "transformers>=4.40" "datasets>=2.19" "accelerate>=0.30" scipy

In [ ]:
# 3. Clone your repo and rebuild the cleaned data on Colab's disk.
#    (Processed CSVs are gitignored, so we regenerate them from source here —
#     one source of truth, nothing large committed to git.)
import os, shutil
if os.path.exists("humor-intelligence"):
    shutil.rmtree("humor-intelligence")
!git clone -q $GITHUB_REPO_URL
%cd humor-intelligence
!python src/data.py
%cd ..

In [ ]:
# 4. Load the cleaned splits
import pandas as pd
BASE = "humor-intelligence/data/processed"
train = pd.read_csv(f"{BASE}/train.csv")
dev   = pd.read_csv(f"{BASE}/dev.csv")
test  = pd.read_csv(f"{BASE}/test.csv")
print(f"train {len(train):,} | dev {len(dev):,} | test {len(test):,}")
train.head(3)

In [ ]:
# 5. Tokenize. Regression needs FLOAT labels.
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(df):
    ds = Dataset.from_pandas(
        df[["joke", "score"]].rename(columns={"score": "labels"}),
        preserve_index=False,
    )
    ds = ds.map(
        lambda b: tokenizer(b["joke"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
    )
    ds = ds.map(lambda ex: {"labels": float(ex["labels"])})
    return ds

train_ds = make_ds(train)
dev_ds   = make_ds(dev)
test_ds  = make_ds(test)
print("tokenized.")

In [ ]:
# 6. Model with a regression head (single output, MSE loss)
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=1, problem_type="regression"
)

In [ ]:
# 7. Spearman metric — the headline number, matches the baseline + the paper
import numpy as np
from scipy.stats import spearmanr, pearsonr

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.asarray(preds).squeeze()
    rho = spearmanr(labels, preds).correlation
    r   = pearsonr(labels, preds)[0]
    return {"spearman": float(rho), "pearson": float(r)}

In [ ]:
# 8. Train. On a free T4, full data x 2 epochs is roughly 1–2 hours.
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

collator = DataCollatorWithPadding(tokenizer)

args = TrainingArguments(
    output_dir="distilbert-humor",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=True,                       # mixed precision — big speedup on T4
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=dev_ds,
    compute_metrics=compute_metrics, data_collator=collator,
)
trainer.train()

In [ ]:
# 9. Final evaluation on the TEST set — this is the number you report
test_metrics = trainer.evaluate(test_ds)
print("TEST Spearman:", round(test_metrics["eval_spearman"], 4))
print("TEST Pearson :", round(test_metrics["eval_pearson"], 4))
print("\nBaseline was 0.363 — did we beat it?")

In [ ]:
# 10. Push the trained model to the Hugging Face Hub.
#     Get a token (write access) at https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
trainer.push_to_hub(HF_MODEL_REPO)
tokenizer.push_to_hub(HF_MODEL_REPO)
print("Pushed to https://huggingface.co/" + HF_MODEL_REPO)

## After this runs

- Note your **test Spearman** — that's your headline result vs. the 0.363 baseline.
- Your model now lives at a public Hugging Face URL you can link in your portfolio.
- Next: pull predictions back locally for a results notebook (confusion matrix,
  predicted-vs-true plot), then build the LLM explanation layer.